# GEDI join and quality filteringReads the subsetted HDF5 files from `02_gedi_download.ipynb`, joins the threeHarmony jobs, removes fill values, clips to the study area, filters, and keepsonly native Cerrado vegetation.| Step | What it does ||------|--------------|| 1 | Setup || 2 | Read the HDF5 files into one table per job || 3 | Join the three tables by `shot_number` || 4 | Remove fill values || 5 | Clip to the exact study area || 6 | Quality filter, one criterion at a time || 7 | Native-vegetation mask (MapBiomas) || 8 | Save footprints and filtering statistics |Run once per year: change `YEAR` in section 1 and re-run.

---## 1. Setup

In [ ]:
from glob import glob
from pathlib import Path
import os

import ee
import h5py
import numpy as np
import pandas as pd
import geopandas as gpd

from agb_cerrado.utils import load_config

cfg = load_config("../config/config.yaml")
ee.Initialize(project=cfg["project"]["gee_project"])

YEAR = 2020
DOWNLOAD_DIR = f"../data/raw/gedi_{YEAR}"
GEOJSON_PATH = "../data/interim/aoi_chapada.geojson"
OUTPUT_PATH = f"../data/interim/gedi_footprints_{YEAR}.parquet"
RESULTS_DIR = Path("../results")

BEAMS = ["BEAM0000", "BEAM0001", "BEAM0010", "BEAM0011",
         "BEAM0101", "BEAM0110", "BEAM1000", "BEAM1011"]
POWER_BEAMS = ["BEAM0101", "BEAM0110", "BEAM1000", "BEAM1011"]

FILL_VALUE = -9999
SENSITIVITY_THRESHOLD = 0.95

MAPBIOMAS_ASSET = ("projects/mapbiomas-public/assets/brazil/lulc/"
                   "collection9/mapbiomas_collection90_integration_v1")
KEEP_CLASSES = cfg["data"]["landcover_keep_classes"]
CHUNK = 3000

files = sorted(glob(os.path.join(DOWNLOAD_DIR, "*_subsetted.h5")))
print(f"Year {YEAR}: {len(files)} files")
print(f"Earth Engine connected")

---## 2. Read the HDF5 filesEach file contains up to eight beam groups. Every variable inside a beam is a1-D array with one value per footprint, so a beam converts directly into aDataFrame. The beam name is kept as a column — that is where the power/coveragedistinction comes from, which is why `beam` was never requested from Harmony.Files are grouped by the set of variables they contain. This recovers the threeHarmony jobs (two L4A halves and L2A) without relying on filenames, which carrya per-granule id rather than a per-job one.

In [ ]:
def read_file(path: str) -> pd.DataFrame:
    """Read all populated beams of one HDF5 file into a single DataFrame."""
    frames = []
    with h5py.File(path, "r") as h5:
        for beam in [b for b in BEAMS if b in h5]:
            variables = list(h5[beam].keys())
            if not variables:
                continue
            df = pd.DataFrame({v: h5[beam][v][:] for v in variables})
            df["beam"] = beam
            frames.append(df)
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()


def variable_signature(path: str) -> tuple:
    """Sorted variable names of the first populated beam, used to group files."""
    with h5py.File(path, "r") as h5:
        for beam in [b for b in BEAMS if b in h5]:
            keys = list(h5[beam].keys())
            if keys:
                return tuple(sorted(keys))
    return ()


groups = {}
for path in files:
    groups.setdefault(variable_signature(path), []).append(path)

tables = []
for signature, group_files in groups.items():
    label = "L2A" if "num_detectedmodes" in signature else "L4A"
    df = pd.concat([read_file(p) for p in group_files], ignore_index=True)
    tables.append(df)
    print(f"{label}: {len(group_files)} files -> {len(df):,} rows x {len(df.columns)} columns")

---## 3. Join by `shot_number``shot_number` is the unique footprint identifier and is identical acrossproducts, so it is the join key.**Duplicated columns.** Harmony always returns `shot_number`, `delta_time` andthe coordinates whether or not they are requested, so those appear in all threetables. `degrade_flag`, `sensitivity` and `solar_elevation` were requested fromboth L4A and L2A deliberately, as a cross-check on the join. Duplicates aredropped against whatever has already been accumulated — a fixed list wouldsilently delete a column that is absent from the first table.Tables are sorted so that L4A comes first. `sensitivity` is therefore kept fromL4A: the target variable is `agbd`, which comes from L4A, so the filter uses thesensitivity of the same product that produced the biomass estimate. The twoproducts compute it with different algorithm settings and disagree on roughly6% of footprints.**The dtype check.** `shot_number` must stay an unsigned integer. Read as floatit loses precision in the last digits and the join fails *silently* — no error,just rows matched to the wrong footprint. The assert turns a silent corruptioninto a loud stop.

In [ ]:
for i, df in enumerate(tables):
    dtype = df["shot_number"].dtype
    print(f"Table {i}: shot_number dtype = {dtype}")
    assert np.issubdtype(dtype, np.integer), "shot_number must stay integer"

# L4A tables first, L2A last (identified by num_detectedmodes)
tables = sorted(tables, key=lambda d: "num_detectedmodes" in d.columns)

merged = tables[0]
for df in tables[1:]:
    dup = [col for col in df.columns if col in merged.columns and col != "shot_number"]
    merged = merged.merge(df.drop(columns=dup), on="shot_number", how="inner")

suffixed = [col for col in merged.columns if col.endswith(("_x", "_y"))]
n_harmony = len(merged)

print(f"\nRows per table: {[len(t) for t in tables]}")
print(f"Rows after join: {n_harmony:,}")
print(f"Duplicate shot_numbers: {merged['shot_number'].duplicated().sum()}")
print(f"Suffixed columns: {suffixed if suffixed else 'none'}")
print(f"Columns: {len(merged.columns)}")

---## 4. Remove fill valuesHarmony returns the full beam structure and writes `-9999` wherever there is novalid measurement. These rows are not footprints. In 2020 they are about half ofall rows, and if left in they corrupt every statistic and every thresholdcomparison — `agbd` averages a negative number and `sensitivity` ranges into thetens of thousands.This must happen before filtering. The count reported here is the honeststarting point of the filtering chain.

In [ ]:
merged = merged.replace(FILL_VALUE, np.nan)
merged = merged.dropna(subset=["agbd", "sensitivity", "elev_lowestmode"])
n_real = len(merged)

print(f"Rows returned by Harmony: {n_harmony:,}")
print(f"Real footprints:          {n_real:,}  ({n_real/n_harmony:.1%})")

print("\nRanges after cleaning:")
for col in ["agbd", "sensitivity", "num_detectedmodes", "elev_lowestmode"]:
    s = merged[col]
    print(f"  {col:<20} min {s.min():>9.2f}   max {s.max():>9.2f}   mean {s.mean():>8.2f}")

---## 5. Clip to the exact study areaHarmony clips to the polygon supplied in the request, so in practice little ornothing is removed here. The step stays as a guarantee: if the service everfalls back to the bounding box, footprints in the corners of that box would beoutside the study area and would silently enter the model.

In [ ]:
aoi = gpd.read_file(GEOJSON_PATH)

gdf = gpd.GeoDataFrame(
    merged,
    geometry=gpd.points_from_xy(merged["lon_lowestmode"], merged["lat_lowestmode"]),
    crs="EPSG:4326",
)

gdf = gdf[gdf.within(aoi.unary_union)].copy()
n_clipped = len(gdf)

print(f"Before clip: {n_real:,}")
print(f"After clip:  {n_clipped:,}  ({n_clipped/n_real:.1%} retained)")

---## 6. Quality filterApplied one criterion at a time so the cost of each is visible. Filtering themall at once would give only the final count; this way the table itself is aresult, and shows which criteria are doing work and which are redundant.| Criterion | Source ||-----------|--------|| `l4_quality_flag == 1` | Kellner et al. (2023) || `l2_quality_flag == 1` | Kellner et al. (2023) || `quality_flag == 1` (L2A) | Beck et al. (2021) || `algorithm_run_flag == 1` | the L4A model ran for this shot || `degrade_flag == 0` | Beck et al. (2021); Moudry et al. (2024) || `num_detectedmodes >= 1` | Moudry et al. (2024) — most effective noise filter || `solar_elevation < 0` | Oliveira et al. (2023) — solar noise in the tropics || power beams only | Beck et al. (2021) || `0 <= agbd <= 500` | artefact removal || `abs(elev_lowestmode - digital_elevation_model) < 100` | Moudry et al. (2024) |The night-only criterion is the one decision that is not settled in theliterature: Moudry et al. (2024) found no day/night difference in temperatesites, while Oliveira et al. (2023) found that solar background noise degradesdaytime acquisitions on sloped terrain in the Brazilian Amazon. The tropicalresult is applied here, since the study area is both tropical and topographicallycomplex.

In [ ]:
filters = [
    ("l4_quality_flag == 1",    lambda d: d["l4_quality_flag"] == 1),
    ("l2_quality_flag == 1",    lambda d: d["l2_quality_flag"] == 1),
    ("quality_flag == 1",       lambda d: d["quality_flag"] == 1),
    ("algorithm_run_flag == 1", lambda d: d["algorithm_run_flag"] == 1),
    ("degrade_flag == 0",       lambda d: d["degrade_flag"] == 0),
    ("num_detectedmodes >= 1",  lambda d: d["num_detectedmodes"] >= 1),
    ("solar_elevation < 0",     lambda d: d["solar_elevation"] < 0),
    ("power beams only",        lambda d: d["beam"].isin(POWER_BEAMS)),
    ("0 <= agbd <= 500",        lambda d: d["agbd"].between(0, 500)),
    ("|elev - DEM| < 100 m",    lambda d: (d["elev_lowestmode"] - d["digital_elevation_model"]).abs() < 100),
]

filtered = gdf.copy()
start = len(filtered)

print(f"{'criterion':<28} {'kept':>10} {'removed':>10} {'% of start':>11}")
print("-" * 62)
print(f"{'(no filter)':<28} {start:>10,} {'':>10} {1.0:>10.1%}")

for label, condition in filters:
    before = len(filtered)
    filtered = filtered[condition(filtered)]
    print(f"{label:<28} {len(filtered):>10,} {before-len(filtered):>10,} {len(filtered)/start:>10.1%}")

### Sensitivity thresholdCompared on the actual data rather than adopted from the literature, since thereference studies are temperate: Moudry et al. (2024) recommend 0.5–0.9 forsparse vegetation and above 0.9 for dense forest, while Fayad et al. (2022) andOliveira et al. (2023) report 0.95–0.98 for dense tropical vegetation. The studyarea spans both regimes.In practice `l4_quality_flag` already enforces a high threshold, so 0.5, 0.9 and0.95 retain the same footprints. Only 0.98 cuts, and it cuts selectively: itremoves roughly three quarters of the data and doubles mean AGBD, becausesensitivity measures penetration through canopy and open physiognomies scorelower by construction, not by poorer quality. Adopting it would bias the sampletowards dense formations and defeat the aim of modelling the full campo limpo –cerradao gradient.

In [ ]:
print(f"Before this filter: {len(filtered):,}\n")
print(f"{'threshold':>10} {'kept':>10} {'% retained':>12} {'mean agbd':>12}")
print("-" * 48)
for t in [0.5, 0.9, 0.95, 0.98]:
    subset = filtered[filtered["sensitivity"] >= t]
    print(f"{t:>10} {len(subset):>10,} {len(subset)/len(filtered):>11.1%} {subset['agbd'].mean():>12.2f}")

filtered = filtered[filtered["sensitivity"] >= SENSITIVITY_THRESHOLD]
n_quality = len(filtered)

print(f"\nApplied: sensitivity >= {SENSITIVITY_THRESHOLD}")
print(f"High-quality footprints: {n_quality:,} ({n_quality/start:.1%} of the clipped set)")

---## 7. Native-vegetation maskFootprints are labelled with their MapBiomas class and restricted to nativeCerrado vegetation. Roughly a fifth fall on pasture, soy or agricultural mosaic— valid measurements, but of a different target than the one being modelled.Zimbres et al. (2021) and Bispo et al. (2020) apply the same mask beforemodelling Cerrado biomass.The class comes from the map of the **footprint's own year**, not a fixedreference year. A parcel cleared in 2021 was still native vegetation in 2020 andits 2020 footprints remain valid. This is what makes the mask consistent withpairing each footprint to predictors from its own year.Classes kept are read from the project config (`landcover_keep_classes`):3 forest formation, 4 savanna formation, 11 wetland, 12 grassland, 29 rockyoutcrop.Sampling runs in chunks because Earth Engine limits the size of a single request.

In [ ]:
classification = ee.Image(MAPBIOMAS_ASSET).select(f"classification_{YEAR}")

labels = []
for start_idx in range(0, len(filtered), CHUNK):
    chunk = filtered.iloc[start_idx:start_idx + CHUNK]
    points = ee.FeatureCollection([
        ee.Feature(ee.Geometry.Point([r.lon_lowestmode, r.lat_lowestmode]))
        for r in chunk.itertuples()
    ])
    sampled = classification.reduceRegions(points, ee.Reducer.first(), 30).getInfo()
    labels.extend(f["properties"].get("first") for f in sampled["features"])
    print(f"  {min(start_idx + CHUNK, len(filtered)):,} / {len(filtered):,}")

filtered = filtered.copy()
filtered["mapbiomas_class"] = labels

native = filtered[filtered["mapbiomas_class"].isin(KEEP_CLASSES)].copy()
removed = filtered[~filtered["mapbiomas_class"].isin(KEEP_CLASSES)]

print(f"\nBefore mask: {n_quality:,}")
print(f"After mask:  {len(native):,}  ({len(native)/n_quality:.1%} retained)")
print(f"\nKept:")
print(native["mapbiomas_class"].value_counts().sort_index().to_string())
print(f"\nRemoved:")
print(removed["mapbiomas_class"].value_counts().sort_index().to_string())

---## 8. SaveTwo outputs, both named by year so that running 2021 and 2022 does not overwritethis one.**The footprints** go to `data/interim/` as Parquet, which preserves dtypes —including the 64-bit `shot_number`, which CSV would return as text or float.**The statistics** go to `results/` as CSV: the filtering chain with counts ateach step, and the MapBiomas class composition of both kept and removedfootprints. These are the numbers the Methods section reports, and they existonly during this run. A later notebook reads the three years back and assemblesthe final tables.

In [ ]:
Path(OUTPUT_PATH).parent.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

out = native.drop(columns="geometry")
out.to_parquet(OUTPUT_PATH, index=False)

print(f"Saved: {OUTPUT_PATH}")
print(f"Footprints: {len(out):,}\n")
print("AGBD (Mg/ha):")
print(out["agbd"].describe().round(2).to_string())

stats = pd.DataFrame([
    {"step": "Records returned by Harmony",           "n": n_harmony},
    {"step": "Real footprints (fill values removed)", "n": n_real},
    {"step": "Within study area",                     "n": n_clipped},
    {"step": "High quality (filter + sensitivity)",   "n": n_quality},
    {"step": "Native vegetation (MapBiomas)",         "n": len(native)},
])
stats["pct_of_start"] = (stats["n"] / n_harmony * 100).round(1)
stats["year"] = YEAR
stats.to_csv(RESULTS_DIR / f"filtering_stats_{YEAR}.csv", index=False)

comp = pd.concat([
    native["mapbiomas_class"].value_counts().rename("n").to_frame().assign(kept=True),
    removed["mapbiomas_class"].value_counts().rename("n").to_frame().assign(kept=False),
])
comp.index.name = "mapbiomas_class"
comp["year"] = YEAR
comp.sort_index().to_csv(RESULTS_DIR / f"mapbiomas_composition_{YEAR}.csv")

print(f"\nStatistics saved to {RESULTS_DIR}/")
print(stats.to_string(index=False))

---## Next steps1. **Other years:** change `YEAR` in section 1 and re-run. The MapBiomas band   follows `YEAR` automatically.2. **Spatial thinning:** footprints are 60 m apart along-track, so neighbouring   ones observe nearly the same vegetation. Thinning is applied per year, before   pooling — a 2021 footprint is not redundant with a 2020 one at the same place,   they are two observations in time.3. **Pooling:** the thinned years are stacked, each footprint keeping its year as   a categorical column and paired with predictors from that same year.### References- Beck, J., Armston, J., Hofton, M., & Luthcke, S. (2021). *GEDI Level 02 User Guide*. USGS EROS.- Bispo, P. C., et al. (2020). Woody aboveground biomass mapping of the Brazilian savanna with a multi-sensor and machine learning approach. *Remote Sensing*, 12(17), 2685.- Fayad, I., Baghdadi, N., & Lahssini, K. (2022). An assessment of the GEDI lasers' capabilities in detecting canopy tops and their penetration in a densely vegetated, tropical area. *Remote Sensing*, 14(13), 2969.- Kellner, J. R., Armston, J., & Duncanson, L. (2023). Algorithm theoretical basis document for GEDI footprint aboveground biomass density. *Earth and Space Science*, 10(4).- Moudry, V., et al. (2024). How to find accurate terrain and canopy height GEDI footprints in temperate forests and grasslands? *Earth and Space Science*, 11.- Oliveira, P. V., Zhang, X., Peterson, B., & Ometto, J. P. (2023). Using simulated GEDI waveforms to evaluate the effects of beam sensitivity and terrain slope on GEDI L2A relative height metrics over the Brazilian Amazon Forest. *Science of Remote Sensing*, 7.- Souza, C. M., et al. (2020). Reconstructing three decades of land use and land cover changes in Brazilian biomes with Landsat archive and Earth Engine. *Remote Sensing*, 12(17), 2735.- Zimbres, B., et al. (2021). Mapping the stock and spatial distribution of aboveground woody biomass in the native vegetation of the Brazilian Cerrado biome. *Forest Ecology and Management*, 499, 119615.